In [1]:
from config_path import add_to_sys_path
add_to_sys_path()  # Call the function to add path

import numpy as np
from sympy.physics.wigner import wigner_3j,wigner_6j
import sympy as sy
from numpy import linalg as LA
from IPython.display import Latex,display
import matplotlib.pyplot as plt
# import seaborn as sns
# sns.set()
# sns.set_palette('terrain')
from Energy_Levels import MoleculeLevels
from molecule_parameters import get_molecule_params
np.set_printoptions(precision=8, suppress=True)

Added C:\Users\Arian\Code\Molecule-Structure\Source Code to sys.path


In [2]:
%matplotlib qt

In [3]:
# port: N extended from [0,1] to [0..4] so select_N([4]) below has states; the larger basis closes the branching sums ([9] -> 1).
X000= MoleculeLevels.initialize_state('YbOH','X','000',[0,1,2,3,4],fermion_or_boson='fermion',isotope='173',M_sublevels='none',I_nuclei=[5/2,0],round=8)
#X000= YbOHLevels.initialize_state('171','X000',[0,1,2,3,4],M_values='none',I=[1/2,0],round=12)

No projection values of J provided, using minimal projection 0.5 as default
Cannot construct decoupled basis without M values


In [4]:
# port: H_odd_A now reads A-state H hyperfine h1/2H, dH, which the odd-YbOH A000 dicts lack (KeyError).
# 0 reproduces the Hamiltonian this notebook was run with (the H terms were added in bacb463, 2026-05-07).
A000= MoleculeLevels.initialize_state('YbOH','A','000',[1],fermion_or_boson='fermion',isotope='173',params={**get_molecule_params('YbOH','173A','000','fermion'),'h1/2H':0,'dH':0},M_sublevels='none',I_nuclei=[5/2,0],round=8)
#A000= YbOHLevels.initialize_state('171','A000',[1,2,3],M_values='none',I=[1/2,0],round=12)

No projection values of J provided, using minimal projection 0.5 as default
Cannot construct decoupled basis without M values


In [5]:
A000.eigensystem(0.0,0.0);
X000.eigensystem(0.0,0.0);

In [6]:
X000.evals0/29979.2458

array([-0.07884176,  0.10966792,  0.39408453,  0.4163821 ,  0.42786627,
        0.58513017,  0.60554877,  0.61383865,  1.37906396,  1.38490625,
        1.39850121,  1.40057423,  1.4168225 ,  1.56903076,  1.57551439,
        1.58927649,  1.58961357,  1.59972195,  2.85227625,  2.85295501,
        2.86284693,  2.86993775,  2.87568844,  2.8869237 ,  2.89337791,
        3.04243033,  3.04344639,  3.05409157,  3.06264095,  3.06587641,
        4.81361828,  4.81477246,  4.82115144,  4.83118116,  4.8326071 ,
        4.84441121,  4.85385865,  5.00395654,  5.00565862,  5.0125737 ,
        5.02483869,  5.02597877])

In [7]:
A000.evals0/29979.2458

array([0.02314353, 0.04899625, 0.46714137, 0.47256818, 0.56665361,
       0.57154823, 0.58240477, 0.59844759, 1.4437158 , 1.44424491,
       1.45091747, 1.46102768])

In [8]:
from Energy_Levels import XA_branching_ratios

In [9]:
BR0 = XA_branching_ratios(X000,A000,0,0.0)

Successfully converted eigenvectors from bBS to aBJ


In [10]:
BR0.sum(axis=0)

array([1.        , 1.00000001, 1.        , 0.99999999, 0.99999999,
       0.99999999, 1.        , 1.        , 1.        , 1.00000002,
       1.        , 0.99999999])

In [11]:
def select_N(N,state):
    idx = []
    N = np.array(N)
    for i,qN in enumerate(state.q_numbers['N']):
        for _N in N:
            if qN == _N:
                idx.append(i)
    return np.array(idx)

def select_J(J,state):
    idx = []
    J = np.array(J)
    for i,qJ in enumerate(state.q_numbers['J']):
        for _J in J:
            if qJ == _J:
                idx.append(i)
    return np.array(idx)

In [12]:
J_idx = A000.select_q({'J':[0.5,1.5]})
print(J_idx)
N_idx = X000.select_q({'N':1})
print(N_idx)

[ 0  1  2  3  4  5  6  7  8  9 10 11]
[2 3 4 5 6 7]


In [13]:
(y,x) = BR0.shape
h=10
fsize = (h,h*x/y)
fig,axes = plt.subplots(1,1,figsize=(15,10),sharey=True,constrained_layout=True)
axes.matshow(BR0)
axes.grid(False)

In [14]:
def gen_state_str(vector,q_numbers,label_q,thresh=0.02,show_coeff=True):
    full_label = r''
    nonzero_idx = np.nonzero(vector)[0]
    first = 0
    for i,index in enumerate(nonzero_idx):
        coeff = vector[index]
        if abs(coeff) < thresh:
            first+=1
            continue
        sign = {True: '+', False: '-'}[coeff > 0]
        sign0 = {True: ' ', False: sign}[sign=='+']
        if show_coeff:
            label_str = r'${}{}|'.format({True: sign0, False: sign}[i==first],abs(coeff))
        else:
            label_str = r'${}|'.format({True: sign0, False: sign}[i==first])
        val = {q:q_numbers[q][index] for q in label_q}
        for q in label_q:
            _q = {True: '\u039B', False: q}[q=='L']
            _q = {True: '\u03A3', False: _q}[q=='Sigma']
            _q = {True: '\u03A9', False: _q}[q=='Omega']
            if (abs(val[q]) % 1) !=0:
                label_str+=r'{}=\frac{{{}}}{{{}}},'.format(_q,*val[q].as_integer_ratio())
            else:
                label_str+=r'{}={},'.format(_q,int(val[q]))
        label_str = label_str[:-1]+r'\rangle$'
        full_label+=label_str
    if i==first and show_coeff==False:
        full_label = r'$'+full_label[2:]
    return full_label

In [15]:
fig,ax = plt.subplots(figsize=(10,10),constrained_layout=True)
# ax.set_aspect('equal')
A_idx = J_idx[-1]+1
X_idx = N_idx[-1]+1
log=False
BR_plot = BR0
if log:
    BR_plot=np.log10(BR0)
    vmin = -10
    vmax = 0
else:
    vmin=None
    vmax = None
#     log_BR = np.copy(BR0)
#     for i,val1 in enumerate(log_BR):
#         for j,val2 in enumerate(val1):
#             if val2 !=0:
#                 log_BR[i,j] = np.log10(val2)
#     BR_plot = log_BR
mat = ax.matshow(BR_plot[:X_idx,:A_idx],cmap=plt.get_cmap('magma'),vmax = vmax, vmin = vmin)
y = np.arange(0,len(X000.evecs0[:X_idx]))
x = np.arange(0,len(A000.evecs0[:A_idx]))
ax.set_yticks(y)
ax.set_xticks(x)
y_labels = [gen_state_str(X000.evecs0[i],X000.q_numbers,[q for q in X000.q_str if (q=='N' or q=='G' or q=='F1')],thresh=0.2,show_coeff=False) for i in range(y.size)]
x_labels = [gen_state_str(A000.evecs0[i],A000.q_numbers,[q for q in A000.q_str if (q=='L' or q=='J' or q=='F1')],thresh=0.2,show_coeff=False) for i in range(x.size)]
ax.set_yticklabels(y_labels, rotation=40, fontsize=12, ha='right')
ax.set_xticklabels(x_labels, rotation=40, fontsize=12,ha='left')
ax.tick_params(axis='both',labelsize=10,direction='out')
ax.grid(True,which='major',color='white',ls='--',linewidth=0.5)
ax.grid(False,which='minor')
(bot,top) = ax.get_ylim()
ax.set_ylim(bot+0.5, top-0.5)
cbar = fig.colorbar(mat, ax=ax,fraction=0.015, pad=0.05)
#ax.set_xlim(-0.5,x.size+0.5);

In [16]:
N_idx = select_N([4],X000)
s = slice(N_idx[0],N_idx[-1]+1)

In [17]:
s

slice(30, 42, None)

In [18]:
BR0[s].sum(axis=0)

array([0.00000829, 0.00000386, 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.00001886, 0.00000931,
       0.00000264, 0.00002167])

In [19]:
def select_N(N,state):
    idx = []
    N = np.array(N)
    for i,qN in enumerate(state.q_numbers['N']):
        for _N in N:
            if qN == _N:
                idx.append(i)
    return np.array(idx)

def select_J(J,state):
    idx = []
    J = np.array(J)
    for i,qJ in enumerate(state.q_numbers['J']):
        for _J in J:
            if qJ == _J:
                idx.append(i)
    return np.array(idx)

In [20]:
J_idx = select_J([0.5],A000)
print(J_idx)
N_idx = select_N([0,2],X000)
print(N_idx)

[0 1 2 3]
[ 0  1  8  9 10 11 12 13 14 15 16 17]


In [21]:
for i in J_idx:
    print(i, A000.evals0[i])
    display(Latex(A000.gen_state_str(i,basis=None,label_q=None,thresh=0.000000001,show_coeff=True,new_line=False,round=16)))

0 693.82557924


<IPython.core.display.Latex object>

1 1468.8707342


<IPython.core.display.Latex object>

2 14004.54596651


<IPython.core.display.Latex object>

3 14167.23755851


<IPython.core.display.Latex object>

In [22]:
for i in N_idx:
    print(i, X000.evals0[i])
    display(Latex(X000.gen_state_str(i,basis='bBS',label_q=None,thresh=0.000001,show_coeff=True,new_line=False,round=None)))

0 -2363.61642029


<IPython.core.display.Latex object>

1 3287.76161146


<IPython.core.display.Latex object>

8 41343.29754469


<IPython.core.display.Latex object>

... (14 outputs left out)

16 47655.41602611


<IPython.core.display.Latex object>

17 47958.4575


<IPython.core.display.Latex object>